In [5]:
import pandas as pd
import numpy as np

df = pd.read_excel("trackman_stolenbases.xlsx", sheet_name="in")
df = df.dropna(subset=["PopTime", "ExchangeTime", "ThrowSpeed"])

print(df.shape)
print(df[["Catcher", "PopTime", "ExchangeTime", "ThrowSpeed"]].head())

(212, 167)
                Catcher   PopTime  ExchangeTime  ThrowSpeed
0  Collender, Sebastian  1.967112      0.700877    78.08361
2         Collado, Joel  2.080746      0.795443    77.42511
3         Collado, Joel  2.255413      0.729754    68.40738
5  Collender, Sebastian  2.063107      0.647629    72.65972
7     Boardman, Cameron  1.766128      0.864614    73.92757


In [7]:
WEIGHT_POP_TIME = 0.50
WEIGHT_THROW_SPEED = 0.30
WEIGHT_EXCHANGE_TIME = 0.20

In [9]:
lg_pop_mean = df["PopTime"].mean()
lg_pop_std = df["PopTime"].std()

lg_exch_mean = df["ExchangeTime"].mean()
lg_exch_std = df["ExchangeTime"].std()

lg_speed_mean = df["ThrowSpeed"].mean()
lg_speed_std = df["ThrowSpeed"].std()

print("Pop Time:", round(lg_pop_mean, 3), round(lg_pop_std, 3))
print("Exchange Time:", round(lg_exch_mean, 3), round(lg_exch_std, 3))
print("Throw Speed:", round(lg_speed_mean, 3), round(lg_speed_std, 3))

Pop Time: 2.0 0.216
Exchange Time: 0.795 0.126
Throw Speed: 77.302 3.566


In [11]:
agg = df.groupby("Catcher").agg(
    pop_time=("PopTime", "mean"),
    exchange_time=("ExchangeTime", "mean"),
    throw_speed=("ThrowSpeed", "mean"),
    attempts=("PopTime", "count")
).reset_index()

print(agg.shape)

(33, 5)


In [13]:
agg["z_pop"] = (lg_pop_mean - agg["pop_time"]) / lg_pop_std
agg["z_exchange"] = (lg_exch_mean - agg["exchange_time"]) / lg_exch_std
agg["z_speed"] = (agg["throw_speed"] - lg_speed_mean) / lg_speed_std

print(agg[["Catcher", "z_pop", "z_exchange", "z_speed"]])

                 Catcher     z_pop  z_exchange   z_speed
0      Applefield, Kalen -0.057492   -0.244677  0.002947
1            Arias, Owen -0.326301    0.620559 -0.777980
2         Belcher, Nolan -0.106224    0.167928  0.794103
3      Boardman, Cameron  0.023724   -0.553655 -0.544409
4           Broski, Adam  0.285671   -0.195362  1.113421
5          Cancel, Chris  0.926820    0.695745 -0.219412
6          Collado, Joel -0.781097    0.258015 -1.229975
7   Collender, Sebastian -0.522501    0.027728 -0.489716
8          DeMastrie, AJ -0.619512   -0.184875 -0.240854
9         Donahue, Scott  0.016511    0.403788 -0.546209
10         Edmunds, Jack -0.174882    0.349194  0.635863
11     Fernandez, Samuel  0.446328    0.313540  1.170311
12           Gell, Scott  2.179058    0.873814  0.246271
13       Haller, Camaron  0.610205    0.728524  0.295650
14   Leighton, Nathaniel  1.328664   -0.308008 -0.894886
15         Leikus, Danny -0.742687   -0.789973 -1.432110
16          McGee, James  0.752

In [15]:
agg["composite_z"] = (
    WEIGHT_POP_TIME * agg["z_pop"] +
    WEIGHT_THROW_SPEED * agg["z_speed"] +
    WEIGHT_EXCHANGE_TIME * agg["z_exchange"]
)

print(agg[["Catcher", "composite_z"]])

                 Catcher  composite_z
0      Applefield, Kalen    -0.076797
1            Arias, Owen    -0.272433
2         Belcher, Nolan     0.218704
3      Boardman, Cameron    -0.262192
4           Broski, Adam     0.437790
5          Cancel, Chris     0.536736
6          Collado, Joel    -0.707938
7   Collender, Sebastian    -0.402620
8          DeMastrie, AJ    -0.418987
9         Donahue, Scott    -0.074850
10         Edmunds, Jack     0.173157
11     Fernandez, Samuel     0.636965
12           Gell, Scott     1.338173
13       Haller, Camaron     0.539502
14   Leighton, Nathaniel     0.334264
15         Leikus, Danny    -0.958971
16          McGee, James     0.740595
17         Mowry, Collin     0.240089
18         Novak, Jayden     0.161179
19      Papetti, Cameron     0.852279
20           Parsons, JJ    -0.574560
21    Pierzynski, Austin     0.366088
22      Polizzi, Stephen    -0.741828
23       Quagliato, Nick    -0.844905
24          Reader, Troy    -0.920845
25         R

In [17]:
agg["backstop_plus"] = (100 + agg["composite_z"] * 15).round(1)

agg = agg.sort_values("backstop_plus", ascending=False).reset_index(drop=True)

print(agg[["Catcher", "pop_time", "exchange_time", "throw_speed", "backstop_plus"]])

                 Catcher  pop_time  exchange_time  throw_speed  backstop_plus
0            Gell, Scott  1.529306       0.685312    78.180260          120.1
1         Russell, Devin  1.880558       0.670438    82.442845          113.6
2       Papetti, Cameron  1.879017       0.679066    81.917880          112.8
3           McGee, James  1.837197       0.669829    79.263706          111.1
4      Fernandez, Samuel  1.903213       0.755630    81.475210          109.6
5          Cancel, Chris  1.799527       0.707661    76.519722          108.1
6        Haller, Camaron  1.867850       0.703547    78.356336          108.1
7           Broski, Adam  1.937881       0.819500    81.272352          106.6
8     Pierzynski, Austin  1.997283       0.774025    81.194723          105.5
9    Leighton, Nathaniel  1.712813       0.833638    74.111110          105.0
10        Schuyler, T.J.  1.993605       0.719923    79.503995          104.8
11          Smith, Avery  1.996928       0.847949    81.182642  

In [19]:
MIN_ATTEMPTS = 6

qualified = agg[agg["attempts"] >= MIN_ATTEMPTS].reset_index(drop=True)
print(qualified[["Catcher", "attempts", "backstop_plus"]])

                 Catcher  attempts  backstop_plus
0          Cancel, Chris        12          108.1
1        Haller, Camaron        14          108.1
2           Broski, Adam        17          106.6
3     Pierzynski, Austin         6          105.5
4           Smith, Avery        12          103.7
5          Mowry, Collin         7          103.6
6         Belcher, Nolan         6          103.3
7          Novak, Jayden         6          102.4
8    Sengenberger, Aidan        10          101.3
9         Donahue, Scott        12           98.9
10     Applefield, Kalen         8           98.8
11     Boardman, Cameron         7           96.1
12           Arias, Owen        10           95.9
13  Collender, Sebastian         6           94.0
14           Parsons, JJ        12           91.4
15      Polizzi, Stephen         6           88.9
16       Quagliato, Nick         6           87.3
17          Reader, Troy         6           86.2
18         Leikus, Danny         7           85.6


In [21]:
print(agg[["Catcher", "attempts"]].sort_values("attempts", ascending=False))

                 Catcher  attempts
7           Broski, Adam        17
6        Haller, Camaron        14
24           Parsons, JJ        12
5          Cancel, Chris        12
11          Smith, Avery        12
17        Donahue, Scott        12
16   Sengenberger, Aidan        10
20           Arias, Owen        10
18     Applefield, Kalen         8
31         Leikus, Danny         7
19     Boardman, Cameron         7
12         Mowry, Collin         7
15         Novak, Jayden         6
22  Collender, Sebastian         6
28      Polizzi, Stephen         6
13        Belcher, Nolan         6
29       Quagliato, Nick         6
30          Reader, Troy         6
8     Pierzynski, Austin         6
4      Fernandez, Samuel         5
3           McGee, James         5
1         Russell, Devin         4
26     Staubley, Brennan         4
10        Schuyler, T.J.         4
23         DeMastrie, AJ         4
27         Collado, Joel         4
0            Gell, Scott         2
25         Righi, Ca